In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_parquet('/kaggle/input/task-f/Рутуб v.2/train.parquet')
test = pd.read_parquet('/kaggle/input/task-f/Рутуб v.2/test.parquet')

In [3]:
def reduce_mem_usage(df):
    """ iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype.name

        if col_type not in ['object', 'category', 'datetime64[ns, UTC]']:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))

    return df

In [ ]:
# Обработка дат

In [6]:
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

Memory usage of dataframe is 4994.57 MB
Memory usage after optimization is: 2537.78 MB
Decreased by 49.2%
Memory usage of dataframe is 460.58 MB
Memory usage after optimization is: 226.47 MB
Decreased by 50.8%


In [ ]:
# Feature Engineering

In [37]:
X = train.drop(columns=['target', 'id', ])
y = train['target']
X_test = test.drop(columns=[  'id',])

In [39]:
from category_encoders import TargetEncoder
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

for i in tqdm(cat_cols):
    te = TargetEncoder(handle_unknown='ignore')
    X[i] = te.fit_transform(X[i], y)
    X_test[i] = te.transform(X_test[i])

100%|██████████| 4/4 [00:36<00:00,  9.09s/it]


In [41]:
X[np.isinf(X)] = np.nan
X_test[np.isinf(X_test)] = np.nan

In [ ]:
from tabm import TabM
from sklearn.preprocessing import StandardScaler
import torch
from sklearn.metrics import f1_score

def train_tabm(X_train, y_train, X_val, y_val, X_test, cat_cols, device='cuda', fold=1, LR=1e-3):
    num_feats = X_train.shape[1] - len(cat_cols)
    cat_cardinalities = [int(X_train[col].nunique()) for col in cat_cols]

    sc = StandardScaler()
    num_cols = [i for i in X_train.columns.tolist() if i not in cat_cols]
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()
    X_train[num_cols] = X_train[num_cols].fillna(X_train[num_cols].mean())
    X_val[num_cols] = X_val[num_cols].fillna(X_train[num_cols].mean())
    X_test[num_cols] = X_test[num_cols].fillna(X_train[num_cols].mean())
    X_train[num_cols] = sc.fit_transform(X_train[num_cols])
    X_val[num_cols] = sc.transform(X_val[num_cols])
    X_test[num_cols] = sc.transform(X_test[num_cols])

    model = TabM.make(
        n_num_features=num_feats,
        cat_cardinalities=cat_cardinalities,
        d_out=len(np.unique(y_train)),
        n_blocks=4,
        d_block=256
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    criterion = torch.nn.CrossEntropyLoss()

    X_train_num = torch.tensor(X_train.drop(columns=cat_cols).values, dtype=torch.float32, device=device)
    X_train_cat = torch.tensor(X_train[cat_cols].values, dtype=torch.long, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)

    X_val_num = torch.tensor(X_val.drop(columns=cat_cols).values, dtype=torch.float32, device=device)
    X_val_cat = torch.tensor(X_val[cat_cols].values, dtype=torch.long, device=device)
    y_val_t = torch.tensor(y_val, dtype=torch.long, device=device)

    train_ds = TensorDataset(X_train_num, X_train_cat, y_train_t)
    val_ds = TensorDataset(X_val_num, X_val_cat, y_val_t)
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

    best_val_score = 0.0
    best_model_state = None
    best_val_preds = None

    for epoch in range(1, 101):
        model.train()
        for xb_num, xb_cat, yb in train_loader:
            optimizer.zero_grad()
            preds_k = model(xb_num, xb_cat)
            logits_mean = preds_k.mean(dim=1)
            loss = criterion(logits_mean, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        all_true = []
        all_logits = []
        with torch.no_grad():
            for xb_num, xb_cat, yb in val_loader:
                preds_k = model(xb_num, xb_cat)
                logits_mean = preds_k.mean(dim=1)
                all_logits.extend(logits_mean.cpu().numpy())
                pred_labels = torch.argmax(logits_mean, dim=1)
                all_preds.append(pred_labels.cpu().numpy())
                all_true.append(yb.cpu().numpy())
        all_preds = np.concatenate(all_preds)
        all_true = np.concatenate(all_true)
        val_f1 = f1_score(all_true, all_preds, average='macro')
        print(f"Epoch {epoch}: TabM val_macro_f1 = {val_f1:.4f}")

        if val_f1 > best_val_score:
            best_val_score = val_f1
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            best_val_preds = all_logits

    print(f"Fold {fold + 1} TabM best_val_score = {best_val_score:.4f}")

    model.load_state_dict(best_model_state)
    model.eval()

    X_test_num = torch.tensor(X_test.drop(columns=cat_cols).values, dtype=torch.float32, device=device)
    X_test_cat = torch.tensor(X_test[cat_cols].values, dtype=torch.long, device=device)
    test_ds = TensorDataset(X_test_num, X_test_cat)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    tabm_test_probs = []
    with torch.no_grad():
        for xb_num, xb_cat in test_loader:
            preds_k = model(xb_num, xb_cat)
            probs_mean = torch.softmax(preds_k.mean(dim=1), dim=1)
            tabm_test_probs.append(probs_mean.cpu().numpy())

    tabm_test_probs = np.concatenate(tabm_test_probs, axis=0)

    return tabm_test_probs, best_val_preds, best_val_score

In [45]:
# Обучение моделей
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score, f1_score

# Инициализация массива для предсказаний на тестовых данных
test_preds_cb = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))
test_preds_lgb = np.zeros(len(X_test))

lgb_params = {
    'n_estimators': 3000,
    'max_depth': 6,
    'learning_rate': 0.07,
    'num_leaves': 60,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'device': 'gpu',
    'early_stopping_round': 100,
    'metric': 'logloss_binary',
    'random_state': 1905,
    'objective': 'binary'
}

cb_params = {
    'iterations': 4000,
    'max_depth': 10,
    'learning_rate': 0.001,
    'task_type': 'GPU',
    'early_stopping_rounds': 100,
    'eval_metric': 'F1',
    'random_state': 1905
}


xgb_params = {
    'n_estimators': 3000,
    'max_depth': 6,
    'learning_rate': 0.1,
    # 'subsample': 0.8,
    # 'colsample_bytree': 0.8,
    # 'gamma': 0.1,
    # 'reg_alpha': 0.5,
    # 'reg_lambda': 0.5,
    'device': 'gpu',
    'early_stopping_rounds': 100,
    'eval_metric': 'logloss',
    'seed': 1905,
    'objective': 'binary:logistic'
}



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1905)
scores_cb = []
scores_xgb = []
scores_lgb = []

for i, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print('#'*15, i+1, '#'*15)
    X_train, X_val, y_train, y_val = X[train_idx], X[val_idx], y[train_idx], y[val_idx]

    # model_cb = CatBoostClassifier(**cb_params)
    # model_cb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    # val_preds_cb = model_cb.predict_proba(X_val)[:, 1]

    model_xgb = XGBClassifier(**xgb_params)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    val_preds_xgb = model_xgb.predict_proba(X_val)[:, 1]

    tabm_preds, tabm_val_preds, tabm_score = train_tabm(X_train, y_train, X_val, y_val, X_test, [], device='cuda', fold=i, LR=1e-3):

    t = 0
    best_score = 0
    best_t = 0
    for i in range(100):
        t = i/100
        preds = (val_preds_xgb > t).astype(int)
        score_cb = f1_score(y_val, preds)
        if best_score < score_cb:
            best_score = score_cb
            best_t = t
    print(f"CatBoost F1 Score: {best_score:.4f}. Best t: {best_t:.4f}")
    scores_cb.append(best_score)
    test_preds_cb += model_xgb.predict_proba(X_test)[:, 1] / 5

    # model_lgb = LGBMClassifier(**lgb_params)
    # model_lgb.fit(X_train, y_train, eval_set=(X_val, y_val), eval_metric='auc')
    # val_preds_lgb = (model_lgb.predict_proba(X_val)[:, 1] > 0.5).astype(int)
    # score_lgb = f1_score(y_val, val_preds_lgb)
    # print(f"LightGBM Recall Score: {score_lgb:.4f}")
    # scores_lgb.append(score_lgb)
    # test_preds_lgb += model_lgb.predict_proba(test)[:, 1] / 5

print('Mean Score:', np.mean(scores_cb))

ensemble_preds = test_preds_cb / 1

############### 1 ###############
[0]	validation_0-logloss:0.66914
[100]	validation_0-logloss:0.54631
[200]	validation_0-logloss:0.54448
[300]	validation_0-logloss:0.54363
[400]	validation_0-logloss:0.54309
[500]	validation_0-logloss:0.54269
[600]	validation_0-logloss:0.54239
[700]	validation_0-logloss:0.54217
[800]	validation_0-logloss:0.54200
[900]	validation_0-logloss:0.54182
[1000]	validation_0-logloss:0.54168
[1100]	validation_0-logloss:0.54156
[1200]	validation_0-logloss:0.54145
[1300]	validation_0-logloss:0.54135
[1400]	validation_0-logloss:0.54127
[1500]	validation_0-logloss:0.54119
[1600]	validation_0-logloss:0.54111
[1700]	validation_0-logloss:0.54104
[1800]	validation_0-logloss:0.54099
[1900]	validation_0-logloss:0.54094
[2000]	validation_0-logloss:0.54089
[2100]	validation_0-logloss:0.54084
[2200]	validation_0-logloss:0.54080
[2300]	validation_0-logloss:0.54077
[2400]	validation_0-logloss:0.54072
[2500]	validation_0-logloss:0.54069
[2600]	validation_0-logloss:0.54066
[2700]

In [43]:
final_preds = (ensemble_preds > 0.38).astype(int)

submit = pd.DataFrame(
    {
        'target': final_preds
    }
)
submit.to_csv('attention_on_screen.csv')

In [44]:
from IPython.display import FileLink

FileLink('attention_on_screen.csv')

/kaggle/working/attention_on_screen.csv